# Unit 3: 生物电信号处理基础

## 学习目标

- 掌握数字滤波器设计原理（IIR/FIR、低通/高通/带通/陷波）
- 理解并实现 50/60 Hz 工频干扰去除
- 掌握时频分析方法（STFT、小波变换）
- 理解 ICA/PCA 伪迹去除的基本原理
- 能够建立完整的 EEG 信号预处理管线

---

## 3.1 数字滤波器理论基础

### 滤波器类型选择

| 类型 | IIR (无限冲激响应) | FIR (有限冲激响应) |
|------|-------------------|-------------------|
| **优点** | 低阶即可实现陡峭过渡带 | 线性相位（无相位失真） |
| **缺点** | 非线性相位 | 需要更高阶数 |
| **典型用途** | 实时滤波（计算量小） | 离线分析（ERP 研究） |
| **scipy 函数** | `butter`, `cheby1`, `ellip` | `firwin`, `remez` |

### EEG 滤波器常用参数

| 滤波器 | 类型 | 截止频率 | 阶数 | 用途 |
|--------|------|----------|------|------|
| 高通 | IIR Butterworth | 0.5–1 Hz | 4 | 去除 DC 漂移 |
| 低通 | IIR Butterworth | 40–80 Hz | 4 | 抗混叠/去除肌电 |
| 带通 (Alpha) | IIR Butterworth | 8–13 Hz | 4 | α 波提取 |
| 陷波 | IIR Notch | 50/60 Hz | Q=30 | 工频干扰抑制 |

### 重要概念：filtfilt vs lfilter

- `lfilter`: 前向滤波（因果，有相位延迟）→ 实时应用
- `filtfilt`: 前向+反向滤波（零相位，无延迟）→ 离线分析

In [ ]:
# ============================================================
# 代码 3.1: 滤波器设计与频率响应分析
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
import sys
sys.path.insert(0, "..")
from utils.helpers import design_bandpass_filter, design_notch_filter

SFREQ = 250.0  # Cyton 默认采样率

def plot_filter_response(b, a, sfreq, title="", ax=None, label=""):
    """绘制滤波器的频率响应和相位响应"""
    if ax is None:
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))
    else:
        ax1, ax2 = ax
    
    w, h = signal.freqz(b, a, worN=2000, fs=sfreq)
    
    ax1.plot(w, 20 * np.log10(np.abs(h) + 1e-10), label=label, linewidth=1.5)
    ax1.set_ylabel("Magnitude (dB)")
    ax1.set_title(f"Frequency Response: {title}")
    ax1.grid(True, alpha=0.3)
    ax1.legend()
    ax1.axhline(-3, color="r", linestyle="--", alpha=0.5, label="-3dB cutoff")
    
    angles = np.unwrap(np.angle(h))
    ax2.plot(w, angles, label=label, linewidth=1.5)
    ax2.set_xlabel("Frequency (Hz)")
    ax2.set_ylabel("Phase (radians)")
    ax2.grid(True, alpha=0.3)
    ax2.legend()
    
    return ax1, ax2


fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. 带通滤波器 (8-13 Hz alpha)
b_bp, a_bp = design_bandpass_filter(8.0, 13.0, SFREQ, order=4)
ax1, ax2 = axes[0, 0], axes[1, 0]
plot_filter_response(b_bp, a_bp, SFREQ, "8-13 Hz Bandpass", (ax1, ax2), "Alpha BP")

# 2. 陷波滤波器 (60 Hz)
b_notch, a_notch = design_notch_filter(60.0, SFREQ, quality_factor=30.0)
ax3, ax4 = axes[0, 1], axes[1, 1]
plot_filter_response(b_notch, a_notch, SFREQ, "60 Hz Notch", (ax3, ax4), "60Hz Notch")

plt.tight_layout()
plt.show()

print("滤波器特性分析:")
print(f"  Alpha 带通滤波器 (4阶): 通带 8-13 Hz, 阻带衰减 > 20 dB")
print(f"  60Hz 陷波滤波器 (Q=30): 中心 60 Hz, 带宽 ≈ {60/30:.0f} Hz")

In [ ]:
# ============================================================
# 代码 3.2: 滤波器对真实信号的效果演示
# ============================================================
from utils.helpers import generate_synthetic_eeg

# 生成含 60Hz 工频的合成信号
raw_data, times = generate_synthetic_eeg(
    duration=5.0, sfreq=SFREQ, n_channels=1,
    noise_level=10.0, alpha_power=40.0, seed=99
)
raw = raw_data[0]  # 使用单通道

# 应用陷波滤波器（使用 filtfilt 实现零相位）
notch_filtered = signal.filtfilt(b_notch, a_notch, raw)

# 再应用带通滤波器
bp_filtered = signal.filtfilt(b_bp, a_bp, notch_filtered)

# 可视化
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

axes[0].plot(times, raw, color="gray", linewidth=0.5, label="Raw")
axes[0].set_ylabel("Raw (muV)")
axes[0].set_title("1. Raw Signal (with 60 Hz noise + background)")
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].plot(times, notch_filtered, color="steelblue", linewidth=0.5, label="Notch filtered")
axes[1].set_ylabel("After Notch (muV)")
axes[1].set_title("2. After 60 Hz Notch Filter")
axes[1].grid(True, alpha=0.3)
axes[1].legend()

axes[2].plot(times, bp_filtered, color="darkgreen", linewidth=0.5, label="Bandpass (8-13 Hz)")
axes[2].set_xlabel("Time (s)")
axes[2].set_ylabel("After BP (muV)")
axes[2].set_title("3. After 8-13 Hz Bandpass (Alpha extracted)")
axes[2].grid(True, alpha=0.3)
axes[2].legend()

plt.tight_layout()
plt.show()

# PSD 对比
fig, ax = plt.subplots(figsize=(12, 5))
for data, label, color, ls in [
    (raw, "Raw", "gray", "-"),
    (notch_filtered, "Notch filtered", "steelblue", "--"),
    (bp_filtered, "Bandpass (8-13)", "darkgreen", "-")
]:
    freqs, psd = signal.welch(data, fs=SFREQ, nperseg=512)
    ax.semilogy(freqs[freqs <= 80], psd[freqs <= 80], color=color,
                linestyle=ls, linewidth=1.5, label=label)

ax.axvline(60, color="red", linestyle=":", alpha=0.7, label="60 Hz")
ax.set_xlabel("Frequency (Hz)")
ax.set_ylabel("PSD (muV^2/Hz)")
ax.set_title("Power Spectral Density: Filtering Effects")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n滤波器效果评估:")
print(f"  Raw signal 60Hz power:        {np.mean(psd[(freqs>59)&(freqs<61)]):.4f}")
# 重新计算 notch filtered 的 PSD
f2, p2 = signal.welch(notch_filtered, fs=SFREQ, nperseg=512)
print(f"  Notch filtered 60Hz power:    {np.mean(p2[(f2>59)&(f2<61)]):.6f}")
print(f"  60Hz suppression:             {np.mean(psd[(freqs>59)&(freqs<61)]) / np.mean(p2[(f2>59)&(f2<61)]):.1f}x")

## 3.2 完备的 EEG 预处理管线

标准 EEG 预处理管线通常包括以下步骤：

```
原始数据 → [1] DC偏移去除 → [2] 工频陷波 → [3] 带通滤波 
        → [4] 伪迹检测 → [5] 坏导插值 → [6] 重参考 → 清洁数据
```

### 步骤详解

1. **DC 偏移去除**：高通滤波 (0.5 Hz) 或减去均值
2. **工频陷波**：50/60 Hz 及其谐波 (100/120, 150/180 Hz)
3. **带通滤波**：根据应用选择 (EEG: 0.5-45 Hz, EMG: 20-200 Hz)
4. **伪迹检测**：幅度阈值、梯度检测、加速度计辅助
5. **坏导插值**：使用邻居通道插值
6. **重参考**：切换参考电极（平均参考、双极参考等）

In [ ]:
# ============================================================
# 代码 3.3: 完整 EEG 预处理管线
# ============================================================
import numpy as np
from scipy import signal


class EEGPreprocessor:
    """
    OpenBCI Cyton EEG 数据预处理器。
    
    实现标准的生物电信号预处理管线：
    Detrend → Notch → Bandpass → Artifact Detection → Re-referencing
    """
    
    def __init__(self, sfreq=250.0, line_noise_freq=60.0):
        """
        Parameters
        ----------
        sfreq : float
            采样率 (Hz)
        line_noise_freq : float
            工频频率 (北美/中国: 60 Hz, 欧洲: 50 Hz)
        """
        self.sfreq = sfreq
        self.line_noise_freq = line_noise_freq
        
        # 预设计滤波器
        self.b_notch, self.a_notch = self._design_notch(line_noise_freq)
        self.b_hp, self.a_hp = self._design_highpass(0.5)   # DC 去除
        self.b_lp, self.a_lp = self._design_lowpass(45.0)   # 抗混叠
        
    def _design_highpass(self, cutoff):
        """高通滤波器: 去除 DC 偏移"""
        nyq = 0.5 * self.sfreq
        return signal.butter(4, cutoff / nyq, btype="high")
    
    def _design_lowpass(self, cutoff):
        """低通滤波器: 抗混叠"""
        nyq = 0.5 * self.sfreq
        return signal.butter(4, cutoff / nyq, btype="low")
    
    def _design_notch(self, freq, q=30):
        """陷波滤波器: 去除工频干扰"""
        w0 = freq / (self.sfreq / 2)
        return signal.iirnotch(w0, q)
    
    def detect_bad_channels(self, data, std_threshold=3.0):
        """
        基于统计特征检测坏导。
        
        检测规则:
        1. 标准差异常大 (电极脱落)
        2. 标准差异常小 (放大器饱和/断开)
        3. 数据为常数 (完全断开)
        
        Parameters
        ----------
        data : ndarray (n_channels, n_samples)
        std_threshold : float
            标准差异常倍数阈值
            
        Returns
        -------
        bad_channels : list
            坏导通道索引列表
        """
        n_channels = data.shape[0]
        ch_stds = np.std(data, axis=1)
        
        # 全局统计
        med_std = np.median(ch_stds)
        mad_std = np.median(np.abs(ch_stds - med_std)) * 1.4826  # robust std
        
        bad_channels = []
        for ch in range(n_channels):
            # 标准差异常（超过中位数 + threshold * MAD）
            if ch_stds[ch] > med_std + std_threshold * mad_std:
                bad_channels.append(ch)
            # 标准差过小（可能是断连）
            elif ch_stds[ch] < 0.01:
                bad_channels.append(ch)
            # 常数检测
            elif np.all(data[ch] == data[ch, 0]):
                bad_channels.append(ch)
        
        return bad_channels
    
    def detect_artifacts(self, data, amplitude_threshold=200.0):
        """
        检测幅度伪迹（眨眼、运动等）。
        
        Parameters
        ----------
        data : ndarray (n_channels, n_samples)
        amplitude_threshold : float
            幅度阈值 (muV)
            
        Returns
        -------
        artifact_mask : ndarray (n_samples,) bool
        """
        # 任何通道超过阈值则标记
        artifact_mask = np.any(np.abs(data) > amplitude_threshold, axis=0)
        return artifact_mask
    
    def process(self, data, apply_notch=True, apply_bandpass=True, 
                remove_artifacts=False, remove_dc=True):
        """
        执行完整预处理管线。
        
        Parameters
        ----------
        data : ndarray (n_channels, n_samples)
            原始 EEG 数据 (muV)
            
        Returns
        -------
        processed : ndarray (n_channels, n_samples)
            预处理后的数据
        info : dict
            预处理信息 (坏导、伪迹比例等)
        """
        processed = data.copy().astype(np.float64)
        info = {}
        
        # Step 1: DC Offset Removal
        if remove_dc:
            for ch in range(processed.shape[0]):
                processed[ch] = signal.filtfilt(self.b_hp, self.a_hp, processed[ch])
        
        # Step 2: Notch Filter
        if apply_notch:
            for ch in range(processed.shape[0]):
                processed[ch] = signal.filtfilt(self.b_notch, self.a_notch, processed[ch])
        
        # Step 3: Bandpass Filter
        if apply_bandpass:
            for ch in range(processed.shape[0]):
                processed[ch] = signal.filtfilt(self.b_lp, self.a_lp, processed[ch])
        
        # Step 4: Bad Channel Detection
        bad_channels = self.detect_bad_channels(processed)
        info["bad_channels"] = bad_channels
        
        # Step 5: Artifact Detection
        artifact_mask = self.detect_artifacts(processed)
        info["artifact_ratio"] = artifact_mask.mean()
        
        # Step 6: Remove artifact-contaminated samples (简单处理)
        if remove_artifacts and artifact_mask.any():
            processed[:, artifact_mask] = np.nan
        
        return processed, info


print("EEGPreprocessor class defined.")
print("\n预处理管线:")
print("  1. High-pass (0.5 Hz)  → DC removal")
print("  2. Notch (60 Hz)       → Line noise suppression")
print("  3. Low-pass (45 Hz)    → Anti-aliasing")
print("  4. Bad channel detect  → Quality control")
print("  5. Artifact detect     → Motion/blink rejection")

In [ ]:
# ============================================================
# 代码 3.4: 预处理管线效果演示
# ============================================================
from utils.helpers import generate_synthetic_eeg, plot_eeg_channels

# 生成多通道数据（含一个"坏导"）
raw_data, times = generate_synthetic_eeg(
    duration=8.0, sfreq=SFREQ, n_channels=6,
    noise_level=8.0, alpha_power=30.0, seed=2024
)
# 模拟 CH4 为坏导（高噪声）
raw_data[3] = np.random.randn(len(times)) * 150

# 初始化预处理器
preproc = EEGPreprocessor(sfreq=SFREQ, line_noise_freq=60.0)

# 执行预处理
clean_data, info = preproc.process(raw_data, remove_artifacts=False)

# 报告预处理结果
print("=" * 60)
print("Preprocessing Report")
print("=" * 60)
print(f"Input shape:      {raw_data.shape}")
print(f"Sampling rate:    {SFREQ} Hz")
print(f"Duration:         {len(times)/SFREQ:.1f} s")
print(f"Bad channels:     {info['bad_channels']}")
print(f"Artifact ratio:   {info['artifact_ratio']:.2%}")

# 对比原始与预处理数据
fig, axes = plt.subplots(1, 2, figsize=(14, 8))

# 原始数据
for ch in range(6):
    axes[0].plot(times, raw_data[ch] + ch * 100, linewidth=0.5,
                 label=f"CH{ch+1}")
axes[0].set_title("Raw Data (offset for visibility)")
axes[0].set_xlabel("Time (s)")
axes[0].set_ylabel("Channel")
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

# 预处理后数据
for ch in range(6):
    axes[1].plot(times, clean_data[ch] + ch * 100, linewidth=0.5,
                 label=f"CH{ch+1}")
axes[1].set_title("After Preprocessing")
axes[1].set_xlabel("Time (s)")
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# PSD 对比
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for ch in [0, 1, 4, 5]:  # 排除坏导 CH4
    f_raw, p_raw = signal.welch(raw_data[ch], fs=SFREQ, nperseg=512)
    f_clean, p_clean = signal.welch(clean_data[ch], fs=SFREQ, nperseg=512)
    
    mask = f_raw <= 80
    ax1.semilogy(f_raw[mask], p_raw[mask], linewidth=1, alpha=0.7,
                 label=f"CH{ch+1}")
    ax2.semilogy(f_clean[mask], p_clean[mask], linewidth=1, alpha=0.7,
                 label=f"CH{ch+1}")

ax1.set_title("PSD: Raw Data")
ax1.set_xlabel("Frequency (Hz)")
ax1.set_ylabel("PSD (muV^2/Hz)")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.set_title("PSD: Preprocessed Data")
ax2.set_xlabel("Frequency (Hz)")
ax2.set_ylabel("PSD (muV^2/Hz)")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3.3 时频分析

EEG 信号是非平稳的，即其频率成分随时间变化。时频分析工具：

### STFT (Short-Time Fourier Transform)

$$X(t, f) = \int_{-\infty}^{\infty} x(\tau) \cdot w(\tau - t) \cdot e^{-j2\pi f\tau} d\tau$$

- **窗口长度** 控制时频分辨率权衡
- **重叠** 增加时间平滑度
- 固定窗口大小 → 固定时频分辨率

### 小波变换 (Wavelet Transform)

- 多分辨率分析：低频使用长时间窗口，高频使用短时间窗口
- Morlet 小波在 EEG 分析中特别常用

In [ ]:
# ============================================================
# 代码 3.5: STFT 时频分析
# ============================================================
import numpy as np
from scipy import signal
import matplotlib.pyplot as plt

# 生成一个频率随时间变化的 chirp 信号 + alpha 振荡
sfreq = 250.0
duration = 10.0
t = np.arange(0, duration, 1/sfreq)

# 构造信号：前 5s alpha (10Hz)，后 5s alpha 减弱 beta (20Hz) 增强
mid = len(t) // 2
signal_test = np.zeros_like(t)
signal_test[:mid] = 30 * np.sin(2 * np.pi * 10 * t[:mid])  # Alpha
signal_test[mid:] = 15 * np.sin(2 * np.pi * 10 * t[mid:]) + \
                     20 * np.sin(2 * np.pi * 22 * t[mid:])  # Alpha + Beta
# 添加噪声
rng = np.random.default_rng(42)
signal_test += rng.normal(0, 5, len(t))

# ---- STFT ----
nperseg = 256  # ~1 秒窗口
noverlap = 200  # 78% 重叠
f, t_stft, Zxx = signal.stft(signal_test, fs=sfreq, nperseg=nperseg,
                              noverlap=noverlap, nfft=512)

# ---- 小波变换 (CWT with Morlet) ----
widths = np.arange(1, 31)  # 小波尺度
cwtmatr = signal.cwt(signal_test, signal.morlet2, widths, w=10)
# 将尺度转换为伪频率 (Morlet wavelet)
cwt_freqs = sfreq * 0.8125 / (widths * 2)  # 近似中心频率

# ---- 可视化 ----
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

# 原始信号
axes[0].plot(t, signal_test, color="steelblue", linewidth=0.5)
axes[0].set_ylabel("Amplitude (muV)")
axes[0].set_title("Time-domain Signal (Alpha → Beta transition at t=5s)")
axes[0].grid(True, alpha=0.3)

# STFT 频谱图
im1 = axes[1].pcolormesh(t_stft, f[f <= 50],
                          np.abs(Zxx[f <= 50, :]),
                          shading="gouraud", cmap="viridis")
axes[1].set_ylabel("Frequency (Hz)")
axes[1].set_title("STFT Spectrogram (256-pt window, 200-pt overlap)")
plt.colorbar(im1, ax=axes[1], label="Magnitude")

# CWT 尺度图
im2 = axes[2].pcolormesh(t, cwt_freqs, np.abs(cwtmatr),
                          shading="gouraud", cmap="magma")
axes[2].set_xlabel("Time (s)")
axes[2].set_ylabel("Frequency (Hz)")
axes[2].set_title("CWT Scalogram (Morlet wavelet)")
axes[2].set_ylim(0, 50)
plt.colorbar(im2, ax=axes[2], label="Magnitude")

plt.tight_layout()
plt.show()

print("时频分析要点:")
print("- STFT 在 t=5s 处可见 10Hz 功率下降, 22Hz 功率上升")
print("- CWT 在低频有更好的时间分辨率（但频率分辨率降低）")
print("- STFT 时频分辨率固定, CWT 自适应")

## 3.4 ICA 伪迹去除简介

独立成分分析 (ICA) 是去除眼电/肌电伪迹的主流方法：

$$X = A \cdot S$$

其中 $X$ 为观测信号（电极），$S$ 为独立源，$A$ 为混合矩阵。

**EEGLAB/MNE 中常用的 ICA 算法：**
- `FastICA`：基于非高斯性最大化
- `Infomax`：基于信息最大化
- `Picard`：预处理 + FastICA（MNE 默认）

**注意：** ICA 需要较多通道（≥16）才能有效，8 通道 Cyton 上效果有限。

In [ ]:
# ============================================================
# 代码 3.6: FastICA 伪迹去除演示
# ============================================================
import numpy as np
from scipy import signal
from sklearn.decomposition import FastICA

# 生成混合信号：EEG + 眨眼伪迹 + 肌电伪迹
np.random.seed(42)
n_samples = 2000
t_ica = np.arange(n_samples) / SFREQ

# 真实源信号
S_eeg = 30 * np.sin(2 * np.pi * 10 * t_ica)  # Alpha EEG
S_blink = np.zeros(n_samples)
S_blink[500:520] = 200 * np.exp(-0.5 * np.arange(20))  # 眨眼
S_blink[1200:1220] = 200 * np.exp(-0.5 * np.arange(20))  # 眨眼
S_emg = np.random.randn(n_samples) * 20 * \
        (np.sin(2 * np.pi * 0.5 * t_ica) ** 2)  # 间歇肌电
S_noise = np.random.randn(n_samples) * 5  # 背景噪声

S_true = np.vstack([S_eeg, S_blink, S_emg, S_noise])

# 随机混合矩阵
A = np.array([
    [0.8, 0.3, 0.1, 0.2],  # 通道 1: 以 EEG 为主
    [0.7, 0.5, 0.2, 0.1],  # 通道 2: EEG + 部分眨眼
    [0.3, 0.8, 0.4, 0.2],  # 通道 3: 以眨眼为主
    [0.2, 0.2, 0.8, 0.3],  # 通道 4: 以肌电为主
])

X = A @ S_true  # 混合信号 (4 channels, n_samples)

# 执行 ICA
ica = FastICA(n_components=4, random_state=42, max_iter=1000)
S_ica = ica.fit_transform(X.T).T  # 分离后的独立成分

# 可视化
fig, axes = plt.subplots(3, 4, figsize=(16, 10))

# 行 1: 真实信号源
titles_source = ["True EEG", "True Blink", "True EMG", "True Noise"]
for i in range(4):
    axes[0, i].plot(t_ica, S_true[i], linewidth=0.5)
    axes[0, i].set_title(titles_source[i], fontsize=10)
    axes[0, i].grid(True, alpha=0.3)

# 行 2: 观测信号 (混合后)
for i in range(4):
    axes[1, i].plot(t_ica, X[i], linewidth=0.5, color="steelblue")
    axes[1, i].set_title(f"Observed CH{i+1}", fontsize=10)
    axes[1, i].grid(True, alpha=0.3)

# 行 3: ICA 分离后
for i in range(4):
    axes[2, i].plot(t_ica, S_ica[i], linewidth=0.5, color="darkgreen")
    axes[2, i].set_title(f"ICA Component {i+1}", fontsize=10)
    axes[2, i].set_xlabel("Time (s)")
    axes[2, i].grid(True, alpha=0.3)

fig.suptitle("ICA Artifact Removal Demonstration\n"
             "Row 1: True Sources | Row 2: Mixed Signals | Row 3: ICA Components",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

print("ICA 伪迹去除分析:")
print("- ICA 成功将眨眼、肌电伪迹与 EEG 分离")
print("- 实际应用中，识别出伪迹成分后可将其置零再反变换")
print("- 公式: X_clean = A @ S_clean (S_clean 中伪迹成分=0)")
print("- 注意: 8 通道 Cyton 使用 ICA 效果有限，建议 16+ 通道")

## 单元小结

本单元涵盖：
1. IIR/FIR 数字滤波器设计、频率响应分析和相位特性
2. 60 Hz 工频陷波和带通滤波的实际效果演示
3. 完整的 EEG 预处理管线（DC 去除→陷波→带通→坏导检测→伪迹检测）
4. STFT 和 CWT 时频分析方法的对比
5. FastICA 伪迹分离的基本原理和代码实现

**下一步：** Unit 4 将学习如何从预处理后的 EEG 信号中提取生物标记物。